# GenAI Tracing and Evaluation with MLflow 3 

https://mlflow.org/docs/latest/genai/data-model/experiments 


In [1]:
%reload_ext autoreload
%autoreload 2

import getpass
import os
import sys
from pathlib import Path

import openai
from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage


### Secrets and Environment Variables

MLflow:<br>
`MLFLOW_TRACKING_URI`<br>

API Keys:<br>
`OPENAI_API_KEY` (for evaluation)<br>
`MISTRAL_API_KEY` (for model usage)


In [2]:
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"
os.environ["MISTRAL_API_KEY"] = getpass.getpass("Enter MISTRAL API key:")
# We still need OpenAI for evaluation
#os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OPENAI API key (for evaluation):")

### Check connection to MLflow


In [3]:
import mlflow 

# List experiments in MLflow
mlflow.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/162850622287800256', creation_time=1750217679010, experiment_id='162850622287800256', last_update_time=1750217679010, lifecycle_stage='active', name='5-genai-with-mlflow-3', tags={}>,
 <Experiment: artifact_location='mlflow-artifacts:/0', creation_time=1750217571826, experiment_id='0', last_update_time=1750217571826, lifecycle_stage='active', name='Default', tags={}>]

In [4]:
# Set up MLflow experiment
mlflow.set_experiment("5-genai-with-mlflow-3")

<Experiment: artifact_location='mlflow-artifacts:/162850622287800256', creation_time=1750217679010, experiment_id='162850622287800256', last_update_time=1750217679010, lifecycle_stage='active', name='5-genai-with-mlflow-3', tags={}>

# Example 1: Tracing a GenAI Application



https://mlflow.org/docs/latest/genai/getting-started/tracing/tracing-notebook 

## Automatic Tracing of LLM calls

In [5]:
# Enable MLflow's autologging to instrument your application with Tracing
mlflow.openai.autolog()

# Create a Mistral client
mistral_client = MistralClient(api_key=os.environ["MISTRAL_API_KEY"])


# Use the trace decorator to capture the application's entry point
@mlflow.trace
def my_app(input: str):
    # Since we're using Mistral, we need custom instrumentation
    # as mlflow.openai.autolog() won't automatically track Mistral
    with mlflow.start_span(name="mistral.chat.completions") as span:
        messages = [
            ChatMessage(role="system", content="You are a helpful assistant."),
            ChatMessage(role="user", content=input)
        ]

        response = mistral_client.chat(
            model="mistral-tiny",  # Mistral's free model
            messages=messages,
        )

        # Manually log some details to the span
        span.set_attribute("mistral.model", "mistral-tiny")

        return response.choices[0].message.content


my_app(input="What is MLflow?")

'MLflow is an open-source platform for managing the end-to-end machine learning (ML) lifecycle. It allows data scientists and machine learning engineers to streamline their workflows, reproducibly run, and deploy models.\n\nThe MLflow platform comprises four main components:\n\n1. Tracking: To record and visualize experiments, including model metrics, parameters, and artifacts.\n\n2. Projects: To package an entire machine learning project, including code, dependencies, and configuration, into a reusable and shareable format.\n\n3. Models: To serve and manage ML models in production, integrating with popular serving frameworks like TensorFlow Serving, Sagemaker, and Kubeflow.\n\n4. Orchestration: To automate and manage the end-to-end ML workflow, including data preparation, training, evaluation, and deployment, using popular tools like Apache Airflow, Kubeflow Pipelines, and others.\n\nOverall, MLflow simplifies the machine learning lifecycle by providing a consistent framework for coll

Trace(trace_id=79fc557df9d94f1db8daec794c23a723)

## Tracing LangChain🦜⛓️

https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langchain 


In [6]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_mistralai import ChatMistralAI


# Enabling autolog for LangChain will enable trace logging.
mlflow.langchain.autolog()

llm = ChatMistralAI(model="mistral-tiny", temperature=0.7, max_tokens=1000)

prompt_template = PromptTemplate.from_template(
    "Answer the question as if you are {person}, fully embodying their style, wit, personality, and habits of speech. "
    "Emulate their quirks and mannerisms to the best of your ability, embracing their traits—even if they aren't entirely "
    "constructive or inoffensive. The question is: {question}"
)

chain = prompt_template | llm | StrOutputParser()

# Let's test another call
chain.invoke(
    {
        "person": "Linus Torvalds",
        "question": "Can I just set everyone's access to sudo to make things easier?",
    }
)

"Well, alrighty then! Let me tell you something, buddy. You're asking me about giving everyone sudo access, huh? That's like handing out the keys to the kingdom without any locks on the doors!\n\nI mean, sure, it might make things easier for you in the short term. But let me ask you this: Do you want to be the administrator who made the system so easy that anyone can just waltz in and mess things up?\n\nRemember, with great power comes great responsibility. Giving everyone sudo access is akin to throwing a party and inviting everyone, including the people who might accidentally knock over the punch bowl or, worse yet, steal the silverware!\n\nSo, what's the alternative, you ask? Well, why not learn the commands and take the time to understand the system? That way, you'll be more efficient in the long run, and you won't have to worry about someone accidentally breaking things because they thought it was as easy as clicking a button.\n\nNow, I'm not saying you can't do it. I'm just sayin

Trace(trace_id=2e9f342d9a9740cab33728db938b79c4)

## Token Usage Tracking

https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langchain#token-usage-tracking


In [7]:
# Execute the chain defined in the previous example
chain.invoke(
    {
        "person": "Linus Torvalds",
        "question": "Can I just set everyone's access to sudo to make things easier?",
    }
)

# Get the trace object just created
last_trace_id = mlflow.get_last_active_trace_id()
trace = mlflow.get_trace(trace_id=last_trace_id)

# Print the token usage
total_usage = trace.info.token_usage
print("== Total token usage: ==")
print(f"  Input tokens: {total_usage['input_tokens']}")
print(f"  Output tokens: {total_usage['output_tokens']}")
print(f"  Total tokens: {total_usage['total_tokens']}")

# Print the token usage for each LLM call
print("\n== Token usage for each LLM call: ==")
for span in trace.data.spans:
    if usage := span.get_attribute("mlflow.chat.tokenUsage"):
        print(f"{span.name}:")
        print(f"  Input tokens: {usage['input_tokens']}")
        print(f"  Output tokens: {usage['output_tokens']}")
        print(f"  Total tokens: {usage['total_tokens']}")

== Total token usage: ==
  Input tokens: 88
  Output tokens: 374
  Total tokens: 462

== Token usage for each LLM call: ==
ChatMistralAI:
  Input tokens: 88
  Output tokens: 374
  Total tokens: 462


Trace(trace_id=ebe1c76675dc4c599c100b736c7c7df1)

# Example 2:  Tracing LangGraph🦜🕸️

https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langgraph 


In [8]:
from typing import Literal

import mlflow

from langchain_core.messages import AIMessage, ToolCall
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langchain_mistralai import ChatMistralAI
from langgraph.prebuilt import create_react_agent

# Enabling tracing for LangGraph (LangChain)
mlflow.langchain.autolog()


@tool
def get_weather(city: Literal["nyc", "sf"]):
    """Use this to get weather information."""
    if city == "nyc":
        return "It might be cloudy in nyc"
    elif city == "sf":
        return "It's always sunny in sf"


llm = ChatMistralAI(model="mistral-tiny")
tools = [get_weather]
graph = create_react_agent(llm, tools)

# Invoke the graph
result = graph.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf?"}]}
)

Trace(trace_id=a5fba9ca25aa458fa18f26b81ce0f1dd)

# Example 3: Prompt Management

https://mlflow.org/docs/latest/genai/mlflow-3/genai-agent

In [9]:

system_prompt = mlflow.genai.register_prompt(
    name="chatbot_prompt",
    template="You are a chatbot that can answer questions about IT. Answer this question: {{question}}",
    commit_message="Initial version of chatbot",
)

2025/06/18 16:47:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: chatbot_prompt, version 1


In [10]:
from langchain.schema.output_parser import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_mistralai import ChatMistralAI

prompt = ChatPromptTemplate.from_template(system_prompt.to_single_brace_format())
chain = prompt | ChatMistralAI(model="mistral-tiny", temperature=0.7) | StrOutputParser()
question = "What is MLflow?"
print(chain.invoke({"question": question}))
# MLflow is an open-source platform for managing the end-to-end machine learning lifecycle...

MLflow is an open-source platform for managing the end-to-end machine learning (ML) lifecycle, including experimentation, reproducibility, model deployment, and model management. It allows data scientists and ML engineers to standardize and automate their ML workflows, making it easier to collaborate, repeat experiments, and deploy models into production. MLflow provides a unified interface that can be used across different ML frameworks such as TensorFlow, PyTorch, Scikit-learn, and more. It also provides tracking, logging, and packaging features to facilitate version control and collaboration.


Trace(trace_id=ee4f48b2895a47fb93cfac452e26cad7)

In [11]:
# set the active model for linking traces
mlflow.set_active_model(name="langchain_model")

# Enable autologging so that interactive traces from the client are automatically linked to a LoggedModel
mlflow.langchain.autolog()

questions = [
    "What is MLflow Tracking and how does it work?",
    "What is Unity Catalog?",
    "What are user-defined functions (UDFs)?",
]
outputs = []

for question in questions:
    outputs.append(chain.invoke({"question": question}))

# fetch the current active model's id and check traces
active_model_id = mlflow.get_active_model_id()
mlflow.search_traces(model_id=active_model_id)
#                            trace_id                                             trace  ...  assessments                        request_id
# 0  e807ab0a020f4794989a24c84c2892ad  Trace(trace_id=e807ab0a020f4794989a24c84c2892ad)  ...           []  e807ab0a020f4794989a24c84c2892ad
# 1  4eb83e4adb6a4f3494bc5b33aca4e970  Trace(trace_id=4eb83e4adb6a4f3494bc5b33aca4e970)  ...           []  4eb83e4adb6a4f3494bc5b33aca4e970
# 2  42b100851f934c969c352930f699308d  Trace(trace_id=42b100851f934c969c352930f699308d)  ...           []  42b100851f934c969c352930f699308d

2025/06/18 16:47:14 INFO mlflow.tracking.fluent: LoggedModel with name 'langchain_model' does not exist, creating one...
2025/06/18 16:47:14 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-3a60d1eca7324beba64d3b63ef45d4de


,trace_id,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,c05ba547fe754be88efeab6c7d5bd9b0,Trace(trace_id=c05ba547fe754be88efeab6c7d5bd9b0),None,TraceState.OK,1750240038473,2591,{'question': 'What are user-defined functions ...,User-defined functions (UDFs) in the context o...,{'mlflow.modelId': 'm-3a60d1eca7324beba64d3b63...,{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': 'w3EIKGmOCNxy0XZNT0UX9A==', 'spa...",[]
1,44c4efd8d1744329bf7fe21ced5b6614,Trace(trace_id=44c4efd8d1744329bf7fe21ced5b6614),None,TraceState.OK,1750240037346,1062,{'question': 'What is Unity Catalog?'},Unity Catalog is a data management and discove...,{'mlflow.modelId': 'm-3a60d1eca7324beba64d3b63...,{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': '/0mN0Dqj/55gYPC1f6Se9w==', 'spa...",[]
2,762591712a934db18b81a51696201841,Trace(trace_id=762591712a934db18b81a51696201841),None,TraceState.OK,1750240034159,3092,{'question': 'What is MLflow Tracking and how ...,MLflow Tracking is a component of the MLflow o...,{'mlflow.modelId': 'm-3a60d1eca7324beba64d3b63...,{'mlflow.artifactLocation': 'mlflow-artifacts:...,"[{'trace_id': '+KBj/HQIxCdpQ8avHxafAA==', 'spa...",[]


[Trace(trace_id=762591712a934db18b81a51696201841), Trace(trace_id=44c4efd8d1744329bf7fe21ced5b6614), Trace(trace_id=c05ba547fe754be88efeab6c7d5bd9b0)]

# Example 4:Evaluate the agent's performance

In [12]:
# Prepare the eval dataset in a pandas DataFrame
import pandas as pd

eval_df = pd.DataFrame(
    {
        "messages": questions,
        "expected_response": [
            """MLflow Tracking is a key component of the MLflow platform designed to record and manage machine learning experiments. It enables data scientists and engineers to log parameters, code versions, metrics, and artifacts in a systematic way, facilitating experiment tracking and reproducibility.\n\nHow It Works:\n\nAt the heart of MLflow Tracking is the concept of a run, which is an execution of a machine learning code. Each run can log the following:\n\nParameters: Input variables or hyperparameters used in the model (e.g., learning rate, number of trees). Metrics: Quantitative measures to evaluate the model's performance (e.g., accuracy, loss). Artifacts: Output files like models, datasets, or images generated during the run. Source Code: The version of the code or Git commit hash used. These logs are stored in a tracking server, which can be set up locally or on a remote server. The tracking server uses a backend storage (like a database or file system) to keep a record of all runs and their associated data.\n\n Users interact with MLflow Tracking through its APIs available in multiple languages (Python, R, Java, etc.). By invoking these APIs in the code, you can start and end runs, and log data as the experiment progresses. Additionally, MLflow offers autologging capabilities for popular machine learning libraries, automatically capturing relevant parameters and metrics without manual code changes.\n\nThe logged data can be visualized using the MLflow UI, a web-based interface that displays all experiments and runs. This UI allows you to compare runs side-by-side, filter results, and analyze performance metrics over time. It aids in identifying the best models and understanding the impact of different parameters.\n\nBy providing a structured way to record experiments, MLflow Tracking enhances collaboration among team members, ensures transparency, and makes it easier to reproduce results. It integrates seamlessly with other MLflow components like Projects and Model Registry, offering a comprehensive solution for managing the machine learning lifecycle.""",
            """Unity Catalog is a feature in Databricks that allows you to create a centralized inventory of your data assets, such as tables, views, and functions, and share them across different teams and projects. It enables easy discovery, collaboration, and reuse of data assets within your organization.\n\nWith Unity Catalog, you can:\n\n1. Create a single source of truth for your data assets: Unity Catalog acts as a central repository of all your data assets, making it easier to find and access the data you need.\n2. Improve collaboration: By providing a shared inventory of data assets, Unity Catalog enables data scientists, engineers, and other stakeholders to collaborate more effectively.\n3. Foster reuse of data assets: Unity Catalog encourages the reuse of existing data assets, reducing the need to create new assets from scratch and improving overall efficiency.\n4. Enhance data governance: Unity Catalog provides a clear view of data assets, enabling better data governance and compliance.\n\nUnity Catalog is particularly useful in large organizations where data is scattered across different teams, projects, and environments. It helps create a unified view of data assets, making it easier to work with data across different teams and projects.""",
            """User-defined functions (UDFs) in the context of Databricks and Apache Spark are custom functions that you can create to perform specific tasks on your data. These functions are written in a programming language such as Python, Java, Scala, or SQL, and can be used to extend the built-in functionality of Spark.\n\nUDFs can be used to perform complex data transformations, data cleaning, or to apply custom business logic to your data. Once defined, UDFs can be invoked in SQL queries or in DataFrame transformations, allowing you to reuse your custom logic across multiple queries and applications.\n\nTo use UDFs in Databricks, you first need to define them in a supported programming language, and then register them with the SparkSession. Once registered, UDFs can be used in SQL queries or DataFrame transformations like any other built-in function.\n\nHere\'s an example of how to define and register a UDF in Python:\n\n```python\nfrom pyspark.sql.functions import udf\nfrom pyspark.sql.types import IntegerType\n\n# Define the UDF function\ndef multiply_by_two(value):\n    return value * 2\n\n# Register the UDF with the SparkSession\nmultiply_udf = udf(multiply_by_two, IntegerType())\n\n# Use the UDF in a DataFrame transformation\ndata = spark.range(10)\nresult = data.withColumn("multiplied", multiply_udf(data.id))\nresult.show()\n```\n\nIn this example, we define a UDF called `multiply_by_two` that multiplies a given value by two. We then register this UDF with the SparkSession using the `udf` function, and use it in a DataFrame transformation to multiply the `id` column of a DataFrame by two.""",
        ],
        "predictions": outputs,
    }
)

# Start a run to represent the evaluation job
with mlflow.start_run() as evaluation_run:
    eval_dataset = mlflow.data.from_pandas(
        df=eval_df,
        name="eval_dataset",
        targets="expected_response",
        predictions="predictions",
    )
    mlflow.log_input(dataset=eval_dataset)
    # Run the evaluation based on extra metrics
    # Current active model will be automatically used
    result = mlflow.evaluate(
        data=eval_dataset,
        extra_metrics=[
            mlflow.metrics.genai.answer_correctness("openai:/gpt-4o"),
            mlflow.metrics.genai.answer_relevance("openai:/gpt-4o"),
        ],
        # This is needed since answer_correctness looks for 'inputs' field
        evaluator_config={"col_mapping": {"inputs": "messages"}},
    )

result.tables["eval_results_table"]
#                                         messages  ...                  answer_relevance/v1/justification
# 0  What is MLflow Tracking and how does it work?  ...  The output directly addresses the input questi...
# 1                         What is Unity Catalog?  ...  The output is completely irrelevant to the inp...
# 2        What are user-defined functions (UDFs)?  ...  The output directly addresses the input questi...

2025/06/18 16:47:35 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-3a60d1eca7324beba64d3b63ef45d4de
2025/06/18 16:47:35 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/06/18 16:47:35 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...
C:\Users\paypa\OneDrive\Desktop\Project\MlOps\experiments-for-modern-ai-and-mlops\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1/1 [00:00<00:00,  9.40it/s]
C:\Users\paypa\OneDrive\Desktop\Project\MlOps\experiments-for-modern-ai-and-mlops\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3859: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
C:\Users\paypa\OneDrive\Desktop\Project\MlOps\exp

🏃 View run capable-flea-651 at: http://127.0.0.1:5000/#/experiments/162850622287800256/runs/e55e5eaa754d4e259e910d2578c28898
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/162850622287800256


,messages,expected_response,predictions,answer_correctness/v1/score,answer_correctness/v1/justification,answer_relevance/v1/score,answer_relevance/v1/justification
0,What is MLflow Tracking and how does it work?,MLflow Tracking is a key component of the MLfl...,MLflow Tracking is a component of the MLflow o...,NaN,Failed to score model on payload. Error: OpenA...,NaN,Failed to score model on payload. Error: OpenA...
1,What is Unity Catalog?,Unity Catalog is a feature in Databricks that ...,Unity Catalog is a data management and discove...,NaN,Failed to score model on payload. Error: OpenA...,NaN,Failed to score model on payload. Error: OpenA...
2,What are user-defined functions (UDFs)?,User-defined functions (UDFs) in the context o...,User-defined functions (UDFs) in the context o...,NaN,Failed to score model on payload. Error: OpenA...,NaN,Failed to score model on payload. Error: OpenA...
